[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week03.ipynb)

# Week 3: MLPs & Backpropagation
**Introduction to Deep Learning (HIT)** &middot; Part I: Foundations

Multilayer perceptrons; the forward pass; backpropagation mechanics via PyTorch autograd.

**Instructor practice notebook** for the 2-hour practice lesson. Work through the sections below live, running each cell and trying the variations. The student homework is the weekly lab.

### Goals

- Build and train a multilayer perceptron.
- Understand the forward pass and backpropagation.
- Use autograd correctly and verify a gradient by hand.

### Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## 1. Build an MLP
Linear layers plus a nonlinearity. nn.Module bundles parameters and the forward pass.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_in=2, d_hidden=16, d_out=2):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, d_hidden), nn.ReLU(), nn.Linear(d_hidden, d_out))
    def forward(self, x):
        return self.net(x)

model = MLP()
print(model)
print("parameter tensors:", sum(p.numel() for p in model.parameters()))

## 2. Activation functions
The nonlinearity between layers; here is what the common ones look like.

In [ ]:
z = torch.linspace(-5, 5, 100)
for name, fn in [("ReLU", F.relu), ("sigmoid", torch.sigmoid), ("tanh", torch.tanh)]:
    plt.plot(z, fn(z), label=name)
plt.legend(); plt.title("Activation functions"); plt.axhline(0, color="k", lw=0.5); plt.show()

## 3. Why the nonlinearity matters
Without ReLU, two linear layers collapse to one linear map.

In [ ]:
torch.manual_seed(0)
X = torch.randn(400, 2); y = (X[:, 0] * X[:, 1] > 0).long()   # XOR-like, not linearly separable

def fit(model, steps=300):
    o = torch.optim.Adam(model.parameters(), 0.03); f = nn.CrossEntropyLoss()
    for _ in range(steps):
        o.zero_grad(); f(model(X), y).backward(); o.step()
    return (model(X).argmax(1) == y).float().mean().item()

linear = nn.Sequential(nn.Linear(2, 16), nn.Linear(16, 2))      # no activation
print("no nonlinearity  accuracy:", round(fit(linear), 3))
print("with ReLU (MLP)  accuracy:", round(fit(MLP()), 3))

## 4. Forward and backward BY HAND
A 2-layer net on one batch: do the chain rule yourself, then check against autograd.

In [ ]:
torch.manual_seed(0)
xb = torch.randn(5, 3)
W1 = torch.randn(3, 4, requires_grad=True); W2 = torch.randn(4, 1, requires_grad=True)

h = torch.relu(xb @ W1)        # forward
out = h @ W2
loss = (out ** 2).mean()
loss.backward()                # autograd gradients

# by hand: d loss / d W2 = h^T @ (2*out/N)
g = 2 * out / out.numel()
manual_W2 = h.t() @ g
print("autograd vs manual dW2 match:", torch.allclose(W2.grad, manual_W2, atol=1e-5))

## 5. Inspect gradients and zero_grad
backward() fills .grad. Gradients ACCUMULATE, so zero them each step.

In [ ]:
w = torch.tensor([2.0], requires_grad=True)
(w ** 2).sum().backward(); print("after 1st backward (d/dw of w^2 = 2w = 4):", w.grad.item())
(w ** 2).sum().backward(); print("WITHOUT zero_grad (accumulated to 8):", w.grad.item())
w.grad.zero_()
(w ** 2).sum().backward(); print("AFTER zero_grad (back to 4):", w.grad.item())

## 6. Hand-derived gradient vs autograd
Sanity-check autograd on something you can differentiate by hand.

In [ ]:
xs = torch.tensor(3.0, requires_grad=True)
ys = xs ** 3 + 2 * xs          # dy/dx = 3x^2 + 2 -> 29 at x = 3
ys.backward()
print("autograd:", xs.grad.item(), "| by hand 3*x^2+2:", 3 * 3 ** 2 + 2)

**Try it live:** add `gradcheck`. `torch.autograd.gradcheck` compares autograd to a numerical estimate.

In [ ]:
from torch.autograd import gradcheck
gfn = lambda a: (a ** 3 + 2 * a).sum()
ok = gradcheck(gfn, (torch.randn(4, dtype=torch.double, requires_grad=True),))
print("gradcheck passed:", ok)

## 7. Train the MLP and read the loss
Put it together: the four-step loop from week 1, now with a real network.

In [ ]:
model = MLP(); o = torch.optim.Adam(model.parameters(), 0.03); f = nn.CrossEntropyLoss()
hist = []
for _ in range(300):
    o.zero_grad(); l = f(model(X), y); l.backward(); o.step(); hist.append(l.item())
print("final accuracy:", round((model(X).argmax(1) == y).float().mean().item(), 3))
plt.plot(hist); plt.xlabel("step"); plt.ylabel("loss"); plt.title("MLP training"); plt.show()

## 8. Visualize the decision boundary
Evaluate the trained MLP on a grid to see the nonlinear regions it learned.

In [ ]:
g = torch.linspace(-3, 3, 120)
gx, gy = torch.meshgrid(g, g, indexing="xy")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)
with torch.no_grad():
    zone = model(grid).argmax(1).reshape(gx.shape)
plt.contourf(gx, gy, zone, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y, s=8, cmap="coolwarm"); plt.title("MLP decision boundary"); plt.show()

### Mini-exercise
Does width help? Train the MLP with d_hidden in [2, 8, 64] and report the accuracy on the XOR-like task.

*Try it before revealing the solution below.*

**Solution.**

In [ ]:
for h_units in [2, 8, 64]:
    torch.manual_seed(0)
    acc = fit(MLP(d_hidden=h_units))
    print(f"hidden={h_units:3d} -> accuracy {acc:.3f}")

**Recap:** nonlinearity is what makes depth worthwhile. Backprop is the chain rule on the graph that the forward pass built; autograd automates it, but gradients must be zeroed each step.

---
Student materials for this week: the lab handout (`labs/week03.html`) and the curated references (`references/week03.html`) in the course site.